# 🐼 Pandas Cheat Sheet — Learn by Doing
### Evolve Robotics India | Data Science Teaching Series

This notebook is a hands-on cheat sheet for Pandas, built around **one custom dataset**:
a fictional record of student enrollments at a robotics/embedded/AI training institute
(sound familiar? 😉).

**Files used:**
- `course_enrollments.csv` — 180 student enrollment records (main dataset)
- `instructors.csv` — course → instructor lookup table (for merge/join)
- `course_enrollments.xlsx` — same data, Excel format (for read_excel demos)

The dataset is intentionally messy — inconsistent casing, extra whitespace in names,
missing ratings/discounts/completion dates — so every cheat sheet section has
real problems to solve, not toy examples.

**Sections:**
1. Creating DataFrames
2. Reading CSV/Excel
3. Selecting Rows & Columns
4. Filtering
5. Sorting
6. Missing Values
7. GroupBy
8. Merge & Join
9. Pivot Tables
10. String Operations
11. DateTime
12. Apply & Lambda

> 💡 Each section = a quick-reference table + runnable examples. Great for reels: one cell = one clip.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print("pandas:", pd.__version__)

pandas: 2.2.2


---
## 1️⃣ Creating DataFrames

| Method | Syntax | Use case |
|---|---|---|
| From dict | `pd.DataFrame({'col': [...]})` | Most common, column-oriented |
| From list of lists | `pd.DataFrame(data, columns=[...])` | Row-oriented data |
| From list of dicts | `pd.DataFrame([{...}, {...}])` | Records / JSON-like data |
| From NumPy array | `pd.DataFrame(arr, columns=[...])` | Numeric arrays |
| Empty with columns | `pd.DataFrame(columns=[...])` | Build up row by row |


In [2]:
# From a dictionary (column-oriented) — quickest way
mini = pd.DataFrame({
    "course": ["Python for Robotics", "Embedded C", "ROS Basics"],
    "seats": [30, 25, 20],
})
mini

,course,seats
0,Python for Robotics,30
1,Embedded C,25
2,ROS Basics,20


In [3]:
# From a list of dicts (record-oriented) — mirrors JSON/API responses
records = [
    {"course": "IoT with ESP32", "seats": 28},
    {"course": "Computer Vision", "seats": 22},
]
pd.DataFrame(records)

,course,seats
0,IoT with ESP32,28
1,Computer Vision,22


In [4]:
# From a NumPy array
arr = np.arange(6).reshape(3, 2)
pd.DataFrame(arr, columns=["a", "b"])

,a,b
0,0,1
1,2,3
2,4,5


---
## 2️⃣ Reading CSV / Excel

| Task | Syntax |
|---|---|
| Read CSV | `pd.read_csv('file.csv')` |
| Parse dates on load | `pd.read_csv('file.csv', parse_dates=['col'])` |
| Set index column | `pd.read_csv('file.csv', index_col='id')` |
| Read only some columns | `pd.read_csv('file.csv', usecols=['a','b'])` |
| Limit rows | `pd.read_csv('file.csv', nrows=100)` |
| Read Excel sheet | `pd.read_excel('file.xlsx', sheet_name='Sheet1')` |
| List all sheets | `pd.ExcelFile('file.xlsx').sheet_names` |
| Write CSV | `df.to_csv('out.csv', index=False)` |
| Write Excel | `df.to_excel('out.xlsx', index=False)` |


In [5]:
# Load the main dataset — parse the two date columns on the way in
df = pd.read_csv(
    "course_enrollments.csv",
    parse_dates=["enrollment_date", "completion_date"],
)
print(df.shape)
df.head()

(180, 12)


,student_id,student_name,course,category,city,mode,enrollment_date,completion_date,fee,discount_pct,rating,status
0,1001,fathima iqbal,ROS Basics,Robotics,COIMBATORE,Offline,2024-02-16,2024-04-24,8971,15.0,5.0,Completed
1,1002,Jithin Sasi,Deep Learning Foundations,AI/ML,COIMBATORE,Offline,2024-07-19,NaT,11590,0.0,3.1,Dropped
2,1003,midhun nair,Deep Learning Foundations,AI/ML,Kozhikode,Offline,2025-03-22,NaT,10830,5.0,NaN,Ongoing
3,1004,Jithin Nair,ROS Basics,Robotics,Thrissur,Offline,2025-02-24,2025-04-10,8433,0.0,3.1,Completed
4,1005,Jithin Roy,Computer Vision with OpenCV,AI/ML,Chennai,Offline,2024-08-09,2024-09-20,12091,0.0,3.9,Completed


In [6]:
df.dtypes

,0
student_id,int64
student_name,object
course,object
category,object
city,object
mode,object
enrollment_date,datetime64[ns]
completion_date,datetime64[ns]
fee,int64
discount_pct,float64


In [7]:
# Reading Excel — list sheets, then load one
xls = pd.ExcelFile("course_enrollments.xlsx")
print("Sheets:", xls.sheet_names)

instructors_from_excel = pd.read_excel("course_enrollments.xlsx", sheet_name="Instructors")
instructors_from_excel.head()

Sheets: ['Enrollments', 'Instructors']


,course,instructor,experience_years
0,Python for Robotics,Praveen S.,6
1,Embedded C Fundamentals,Anitha R.,4
2,Machine Learning Bootcamp,Vishal K.,5
3,IoT with ESP32,Praveen S.,6
4,ROS Basics,Anitha R.,4


In [8]:
# Also load the lookup CSV we'll use later for merge/join
instructors = pd.read_csv("instructors.csv")
instructors

,course,instructor,experience_years
0,Python for Robotics,Praveen S.,6
1,Embedded C Fundamentals,Anitha R.,4
2,Machine Learning Bootcamp,Vishal K.,5
3,IoT with ESP32,Praveen S.,6
4,ROS Basics,Anitha R.,4
5,Data Science with Python,Vishal K.,5
6,Arduino Bootcamp,Deepak M.,3
7,Computer Vision with OpenCV,Sona T.,7
8,Deep Learning Foundations,Sona T.,7
9,STM32 Embedded Systems,Deepak M.,3


---
## 3️⃣ Selecting Rows & Columns

| Task | Syntax |
|---|---|
| Single column (Series) | `df['col']` |
| Multiple columns (DataFrame) | `df[['col1','col2']]` |
| Row(s) by label | `df.loc[label]` / `df.loc[[l1,l2]]` |
| Row(s) by position | `df.iloc[0]` / `df.iloc[0:5]` |
| Row + column by label | `df.loc[row, col]` |
| Row + column by position | `df.iloc[row_pos, col_pos]` |
| First / last n rows | `df.head(n)` / `df.tail(n)` |


In [9]:
# Single column -> Series
df["course"].head()

,course
0,ROS Basics
1,Deep Learning Foundations
2,Deep Learning Foundations
3,ROS Basics
4,Computer Vision with OpenCV


In [10]:
# Multiple columns -> DataFrame (note the double brackets)
df[["student_name", "course", "fee"]].head()

,student_name,course,fee
0,fathima iqbal,ROS Basics,8971
1,Jithin Sasi,Deep Learning Foundations,11590
2,midhun nair,Deep Learning Foundations,10830
3,Jithin Nair,ROS Basics,8433
4,Jithin Roy,Computer Vision with OpenCV,12091


In [11]:
# Rows by position
df.iloc[0:3]

,student_id,student_name,course,category,city,mode,enrollment_date,completion_date,fee,discount_pct,rating,status
0,1001,fathima iqbal,ROS Basics,Robotics,COIMBATORE,Offline,2024-02-16,2024-04-24,8971,15.0,5.0,Completed
1,1002,Jithin Sasi,Deep Learning Foundations,AI/ML,COIMBATORE,Offline,2024-07-19,NaT,11590,0.0,3.1,Dropped
2,1003,midhun nair,Deep Learning Foundations,AI/ML,Kozhikode,Offline,2025-03-22,NaT,10830,5.0,NaN,Ongoing


In [12]:
# Specific cell(s): rows 0-2, only 'student_name' and 'rating' columns
df.loc[0:2, ["student_name", "rating"]]

,student_name,rating
0,fathima iqbal,5.0
1,Jithin Sasi,3.1
2,midhun nair,NaN


In [13]:
# .at / .iat for fast single-value access
print(df.at[0, "student_name"])
print(df.iat[0, 0])

fathima iqbal
1001


---
## 4️⃣ Filtering

| Task | Syntax |
|---|---|
| Single condition | `df[df['col'] > x]` |
| Multiple conditions (AND) | `df[(cond1) & (cond2)]` |
| Multiple conditions (OR) | `df[(cond1) \| (cond2)]` |
| Membership | `df[df['col'].isin([...])]` |
| Negation | `df[~cond]` |
| Range | `df[df['col'].between(a, b)]` |
| Readable syntax | `df.query("col > x and other == 'y'")` |
| String contains | `df[df['col'].str.contains('x')]` |


In [14]:
# Single condition: high-fee enrollments
df[df["fee"] > 10000].head()

,student_id,student_name,course,category,city,mode,enrollment_date,completion_date,fee,discount_pct,rating,status
1,1002,Jithin Sasi,Deep Learning Foundations,AI/ML,COIMBATORE,Offline,2024-07-19,NaT,11590,0.0,3.1,Dropped
2,1003,midhun nair,Deep Learning Foundations,AI/ML,Kozhikode,Offline,2025-03-22,NaT,10830,5.0,NaN,Ongoing
4,1005,Jithin Roy,Computer Vision with OpenCV,AI/ML,Chennai,Offline,2024-08-09,2024-09-20,12091,0.0,3.9,Completed
6,1007,PARVATHY CHANDRAN,Deep Learning Foundations,AI/ML,Thrissur,Online,2025-04-29,NaT,11076,0.0,NaN,Ongoing
9,1010,Nandana Mohan,Computer Vision with OpenCV,AI/ML,Online,Online,2024-12-08,2025-01-03,11618,0.0,3.6,Completed


In [15]:
# Multiple conditions: Robotics category AND Completed status
robotics_completed = df[(df["category"] == "Robotics") & (df["status"] == "Completed")]
robotics_completed.shape

(25, 12)

In [16]:
# isin() for membership checks
target_courses = ["Python for Robotics", "IoT with ESP32", "ROS Basics"]
df[df["course"].isin(target_courses)].head()

,student_id,student_name,course,category,city,mode,enrollment_date,completion_date,fee,discount_pct,rating,status
0,1001,fathima iqbal,ROS Basics,Robotics,COIMBATORE,Offline,2024-02-16,2024-04-24,8971,15.0,5.0,Completed
3,1004,Jithin Nair,ROS Basics,Robotics,Thrissur,Offline,2025-02-24,2025-04-10,8433,0.0,3.1,Completed
5,1006,SARATH ROY,IoT with ESP32,Embedded,Bangalore,Offline,2024-12-27,2025-01-27,8165,0.0,4.4,Completed
7,1008,Vivek Varma,Python for Robotics,Robotics,Kochi,Offline,2025-02-25,NaT,8829,15.0,NaN,Ongoing
16,1017,Jithin Krishnan,Python for Robotics,Robotics,COIMBATORE,Offline,2024-03-09,2024-04-29,9117,0.0,3.6,Completed


In [17]:
# .between() for ranges, and .query() for readable filters
df[df["fee"].between(7000, 8000)].head(3)

,student_id,student_name,course,category,city,mode,enrollment_date,completion_date,fee,discount_pct,rating,status
24,1025,meera mohan,IoT with ESP32,Embedded,Chennai,Offline,2024-09-05,2024-11-20,7769,10.0,4.9,Completed
28,1029,Fathima Chandran,Embedded C Fundamentals,Embedded,Kochi,Online,2024-07-17,NaT,7698,0.0,NaN,Dropped
29,1030,GOPIKA IQBAL,STM32 Embedded Systems,Embedded,COIMBATORE,Offline,2024-08-22,2024-09-21,7058,15.0,3.5,Completed


In [18]:
df.query("category == 'AI/ML' and fee > 10000").head(3)

,student_id,student_name,course,category,city,mode,enrollment_date,completion_date,fee,discount_pct,rating,status
1,1002,Jithin Sasi,Deep Learning Foundations,AI/ML,COIMBATORE,Offline,2024-07-19,NaT,11590,0.0,3.1,Dropped
2,1003,midhun nair,Deep Learning Foundations,AI/ML,Kozhikode,Offline,2025-03-22,NaT,10830,5.0,NaN,Ongoing
4,1005,Jithin Roy,Computer Vision with OpenCV,AI/ML,Chennai,Offline,2024-08-09,2024-09-20,12091,0.0,3.9,Completed


---
## 5️⃣ Sorting

| Task | Syntax |
|---|---|
| Sort by one column | `df.sort_values('col')` |
| Descending | `df.sort_values('col', ascending=False)` |
| Sort by multiple columns | `df.sort_values(['col1','col2'], ascending=[True, False])` |
| Sort by index | `df.sort_index()` |
| Get top-n fastest | `df.nlargest(n, 'col')` / `df.nsmallest(n, 'col')` |


In [19]:
# Highest-rated enrollments first
df.sort_values("rating", ascending=False).head()

,student_id,student_name,course,category,city,mode,enrollment_date,completion_date,fee,discount_pct,rating,status
0,1001,fathima iqbal,ROS Basics,Robotics,COIMBATORE,Offline,2024-02-16,2024-04-24,8971,15.0,5.0,Completed
75,1076,Nandana Bose,Machine Learning Bootcamp,Data Science,Kozhikode,Offline,2024-03-05,2024-04-29,10416,15.0,5.0,Completed
179,1180,Kavya Das,Python for Robotics,Robotics,Online,Online,2025-05-09,2025-07-24,8152,5.0,5.0,Completed
24,1025,meera mohan,IoT with ESP32,Embedded,Chennai,Offline,2024-09-05,2024-11-20,7769,10.0,4.9,Completed
20,1021,ANJALI KUMAR,ROS Basics,Robotics,Kochi,Online,2024-02-14,2024-04-20,8481,10.0,4.9,Completed


In [20]:
# Multi-column sort: category A-Z, then fee highest-first within each category
df.sort_values(["category", "fee"], ascending=[True, False]).head(6)

,student_id,student_name,course,category,city,mode,enrollment_date,completion_date,fee,discount_pct,rating,status
31,1032,vivek nair,Deep Learning Foundations,AI/ML,kochi,Online,2024-03-07,NaT,12446,5.0,NaN,Dropped
90,1091,devika mohan,Deep Learning Foundations,AI/ML,Bangalore,Offline,2024-08-28,2024-11-13,12406,0.0,3.7,Completed
148,1149,Rohan Thomas,Deep Learning Foundations,AI/ML,Chennai,Offline,2024-04-16,NaT,12379,0.0,NaN,Ongoing
23,1024,nikhil roy,Computer Vision with OpenCV,AI/ML,Online,Online,2025-02-19,2025-04-23,12332,0.0,3.5,Completed
59,1060,Midhun Pillai,Computer Vision with OpenCV,AI/ML,Thrissur,Online,2024-01-08,NaT,12321,0.0,NaN,Ongoing
111,1112,Karthik Iqbal,Deep Learning Foundations,AI/ML,Bangalore,Online,2024-05-14,2024-08-06,12318,5.0,4.2,Completed


In [21]:
# Quick top-5 without a full sort
df.nlargest(5, "fee")[["student_name", "course", "fee"]]

,student_name,course,fee
31,vivek nair,Deep Learning Foundations,12446
90,devika mohan,Deep Learning Foundations,12406
148,Rohan Thomas,Deep Learning Foundations,12379
23,nikhil roy,Computer Vision with OpenCV,12332
59,Midhun Pillai,Computer Vision with OpenCV,12321


---
## 6️⃣ Missing Values

| Task | Syntax |
|---|---|
| Count missing per column | `df.isna().sum()` |
| Rows with any missing | `df[df.isna().any(axis=1)]` |
| Drop rows with any NaN | `df.dropna()` |
| Drop rows where specific col is NaN | `df.dropna(subset=['col'])` |
| Fill with a constant | `df.fillna(0)` |
| Fill with column mean/median | `df['col'].fillna(df['col'].mean())` |
| Forward/backward fill | `df.fillna(method='ffill')` |
| Check a single value | `pd.isna(x)` |


In [22]:
# Where are the gaps?
df.isna().sum()

,0
student_id,0
student_name,0
course,0
category,0
city,0
mode,0
enrollment_date,0
completion_date,73
fee,0
discount_pct,19


In [23]:
# Ratings are missing for Ongoing/some Dropped students — that's expected, not a data error
df[df["rating"].isna()]["status"].value_counts()

,count
status,
Ongoing,43
Dropped,12


In [24]:
# Fill missing discount_pct with 0 (no discount given), keep a copy so original stays intact
df_clean = df.copy()
df_clean["discount_pct"] = df_clean["discount_pct"].fillna(0)

# Fill missing rating with the course-level average rating (smarter than a global fill)
df_clean["rating"] = df_clean.groupby("course")["rating"].transform(lambda s: s.fillna(s.mean()))

df_clean[["course", "discount_pct", "rating"]].isna().sum()

,0
course,0
discount_pct,0
rating,0


In [25]:
# completion_date is genuinely absent for Ongoing/Dropped students -> leave as NaT, don't fake it
df_clean[df_clean["completion_date"].isna()]["status"].value_counts()

,count
status,
Ongoing,43
Dropped,30


---
## 7️⃣ GroupBy

| Task | Syntax |
|---|---|
| Group + aggregate one col | `df.groupby('col')['other'].mean()` |
| Group + multiple aggs | `df.groupby('col').agg({'a':'mean','b':'sum'})` |
| Group by multiple columns | `df.groupby(['col1','col2'])` |
| Count rows per group | `df.groupby('col').size()` |
| Named aggregations | `df.groupby('col').agg(avg=('a','mean'))` |
| Group + transform (same shape back) | `df.groupby('col')['a'].transform('mean')` |


In [26]:
# Average fee per category
df_clean.groupby("category")["fee"].mean().round(0)

,fee
category,
AI/ML,11609.0
Data Science,9549.0
Embedded,7405.0
Robotics,8522.0


In [27]:
# Multiple stats at once, with clear output names
summary = df_clean.groupby("category").agg(
    avg_fee=("fee", "mean"),
    total_students=("student_id", "count"),
    avg_rating=("rating", "mean"),
).round(2)
summary

,avg_fee,total_students,avg_rating
category,,,
AI/ML,11609.13,39,3.50
Data Science,9549.49,39,3.55
Embedded,7404.59,66,3.94
Robotics,8522.08,36,3.57


In [28]:
# Group by TWO columns: category + status
df_clean.groupby(["category", "status"]).size().unstack(fill_value=0)

status,Completed,Dropped,Ongoing
category,,,
AI/ML,19,9,11
Data Science,22,7,10
Embedded,41,9,16
Robotics,25,5,6


---
## 8️⃣ Merge & Join

| Task | Syntax |
|---|---|
| Inner join (default) | `pd.merge(df1, df2, on='key')` |
| Left join | `pd.merge(df1, df2, on='key', how='left')` |
| Right / outer join | `how='right'` / `how='outer'` |
| Different column names | `pd.merge(df1, df2, left_on='a', right_on='b')` |
| Join on index | `df1.join(df2)` |
| Stack rows (same columns) | `pd.concat([df1, df2])` |
| Stack columns side by side | `pd.concat([df1, df2], axis=1)` |


In [29]:
# Left join: attach instructor info to every enrollment via the 'course' key
enriched = pd.merge(df_clean, instructors, on="course", how="left")
enriched[["student_name", "course", "instructor", "experience_years"]].head()

,student_name,course,instructor,experience_years
0,fathima iqbal,ROS Basics,Anitha R.,4
1,Jithin Sasi,Deep Learning Foundations,Sona T.,7
2,midhun nair,Deep Learning Foundations,Sona T.,7
3,Jithin Nair,ROS Basics,Anitha R.,4
4,Jithin Roy,Computer Vision with OpenCV,Sona T.,7


In [30]:
# Sanity check: any enrollment that didn't find a matching instructor?
enriched["instructor"].isna().sum()

np.int64(0)

In [31]:
# concat() to stack two subsets back together (e.g. two batches of new admissions)
batch_a = df_clean[df_clean["category"] == "Robotics"].head(2)
batch_b = df_clean[df_clean["category"] == "AI/ML"].head(2)
pd.concat([batch_a, batch_b])[["student_name", "category"]]

,student_name,category
0,fathima iqbal,Robotics
3,Jithin Nair,Robotics
1,Jithin Sasi,AI/ML
2,midhun nair,AI/ML


---
## 9️⃣ Pivot Tables

| Task | Syntax |
|---|---|
| Basic pivot | `df.pivot_table(index='a', columns='b', values='c')` |
| Choose aggregation | `pivot_table(..., aggfunc='sum')` |
| Multiple agg funcs | `pivot_table(..., aggfunc=['mean','count'])` |
| Fill empty cells | `pivot_table(..., fill_value=0)` |
| Row/col totals | `pivot_table(..., margins=True)` |
| Quick frequency table | `pd.crosstab(df['a'], df['b'])` |


In [32]:
# Average fee by category x mode, with row/column totals
pd.pivot_table(
    df_clean,
    index="category",
    columns="mode",
    values="fee",
    aggfunc="mean",
    margins=True,
    fill_value=0,
).round(0)

mode,Offline,Online,All
category,,,
AI/ML,11560.0,11647.0,11609.0
Data Science,9700.0,9474.0,9549.0
Embedded,7387.0,7415.0,7405.0
Robotics,8361.0,8625.0,8522.0
All,9073.0,8962.0,9004.0


In [33]:
# crosstab: how many students per category fall into each status?
pd.crosstab(df_clean["category"], df_clean["status"])

status,Completed,Dropped,Ongoing
category,,,
AI/ML,19,9,11
Data Science,22,7,10
Embedded,41,9,16
Robotics,25,5,6


---
## 🔟 String Operations (`.str` accessor)

| Task | Syntax |
|---|---|
| Strip whitespace | `df['col'].str.strip()` |
| Change case | `.str.lower()` / `.str.upper()` / `.str.title()` |
| Contains substring | `.str.contains('x', case=False)` |
| Replace text | `.str.replace('a', 'b')` |
| Split into parts | `.str.split(' ')` |
| String length | `.str.len()` |
| Starts/ends with | `.str.startswith('x')` |
| Extract with regex | `.str.extract(r'(\d+)')` |


In [34]:
# Our student_name and city columns are messy on purpose — classic real-world data
df_clean[["student_name", "city"]].head(6)

,student_name,city
0,fathima iqbal,COIMBATORE
1,Jithin Sasi,COIMBATORE
2,midhun nair,Kozhikode
3,Jithin Nair,Thrissur
4,Jithin Roy,Chennai
5,SARATH ROY,Bangalore


In [35]:
# Clean it up: strip whitespace, then title-case names and cities consistently
df_clean["student_name"] = df_clean["student_name"].str.strip().str.title()
df_clean["city"] = df_clean["city"].str.strip().str.title()

df_clean[["student_name", "city"]].head(6)

,student_name,city
0,Fathima Iqbal,Coimbatore
1,Jithin Sasi,Coimbatore
2,Midhun Nair,Kozhikode
3,Jithin Nair,Thrissur
4,Jithin Roy,Chennai
5,Sarath Roy,Bangalore


In [36]:
# Split full name into first/last
name_parts = df_clean["student_name"].str.split(" ", n=1, expand=True)
df_clean["first_name"] = name_parts[0]
df_clean["last_name"] = name_parts[1]

df_clean[["student_name", "first_name", "last_name"]].head()

,student_name,first_name,last_name
0,Fathima Iqbal,Fathima,Iqbal
1,Jithin Sasi,Jithin,Sasi
2,Midhun Nair,Midhun,Nair
3,Jithin Nair,Jithin,Nair
4,Jithin Roy,Jithin,Roy


In [37]:
# Find everyone whose course mentions 'Embedded' (case-insensitive)
df_clean[df_clean["course"].str.contains("embedded", case=False)]["course"].unique()

array(['Embedded C Fundamentals', 'STM32 Embedded Systems'], dtype=object)

---
## 1️⃣1️⃣ DateTime

| Task | Syntax |
|---|---|
| Convert to datetime | `pd.to_datetime(df['col'])` |
| Extract year/month/day | `df['col'].dt.year` / `.dt.month` / `.dt.day` |
| Day of week | `df['col'].dt.day_name()` |
| Difference between dates | `df['end'] - df['start']` → Timedelta |
| Filter by date range | `df[df['col'].between('2024-01-01','2024-06-30')]` |
| Group by month | `df.groupby(df['col'].dt.to_period('M'))` |
| Set as index for resampling | `df.set_index('col').resample('M')` |


In [38]:
# Already parsed as datetime when we read the CSV — extract useful parts
df_clean["enroll_month"] = df_clean["enrollment_date"].dt.month_name()
df_clean["enroll_weekday"] = df_clean["enrollment_date"].dt.day_name()

df_clean[["enrollment_date", "enroll_month", "enroll_weekday"]].head()

,enrollment_date,enroll_month,enroll_weekday
0,2024-02-16,February,Friday
1,2024-07-19,July,Friday
2,2025-03-22,March,Saturday
3,2025-02-24,February,Monday
4,2024-08-09,August,Friday


In [39]:
# How long did it take completed students to finish? (Timedelta arithmetic)
df_clean["days_to_complete"] = (df_clean["completion_date"] - df_clean["enrollment_date"]).dt.days
df_clean["days_to_complete"].describe()

,days_to_complete
count,107.000000
mean,55.401869
std,18.485007
min,20.000000
25%,41.000000
50%,54.000000
75%,71.500000
max,89.000000


In [40]:
# Enrollments per month, using to_period for clean monthly grouping
monthly = df_clean.groupby(df_clean["enrollment_date"].dt.to_period("M")).size()
monthly.head(12)

,0
enrollment_date,
2024-01,13
2024-02,8
2024-03,11
2024-04,8
2024-05,11
2024-06,5
2024-07,10
2024-08,10
2024-09,6


In [41]:
# Filter to just the first half of 2024
first_half_2024 = df_clean[df_clean["enrollment_date"].between("2024-01-01", "2024-06-30")]
first_half_2024.shape

(56, 17)

---
## 1️⃣2️⃣ Apply & Lambda

| Task | Syntax |
|---|---|
| Apply to a Series | `df['col'].apply(func)` |
| Apply with lambda | `df['col'].apply(lambda x: ...)` |
| Apply row-wise on DataFrame | `df.apply(lambda row: ..., axis=1)` |
| Apply to every element | `df.applymap(func)` (DataFrame-wide) |
| Map values via dict | `df['col'].map({'a': 1, 'b': 2})` |
| Vectorized (preferred when possible) | `np.where(cond, a, b)` |


In [42]:
# Simple Series.apply with a lambda: net fee after discount
df_clean["net_fee"] = df_clean.apply(
    lambda row: row["fee"] * (1 - row["discount_pct"] / 100), axis=1
)
df_clean[["fee", "discount_pct", "net_fee"]].head()

,fee,discount_pct,net_fee
0,8971,15.0,7625.35
1,11590,0.0,11590.00
2,10830,5.0,10288.50
3,8433,0.0,8433.00
4,12091,0.0,12091.00


In [43]:
# A named function is often clearer than a long lambda
def rating_band(r):
    if pd.isna(r):
        return "No rating"
    elif r >= 4.5:
        return "Excellent"
    elif r >= 3.5:
        return "Good"
    else:
        return "Needs improvement"

df_clean["rating_band"] = df_clean["rating"].apply(rating_band)
df_clean["rating_band"].value_counts()

,count
rating_band,
Good,88
Needs improvement,64
Excellent,28


In [44]:
# map() for quick value substitution via a dictionary
mode_short = {"Online": "ON", "Offline": "OFF"}
df_clean["mode_code"] = df_clean["mode"].map(mode_short)
df_clean[["mode", "mode_code"]].head()

,mode,mode_code
0,Offline,OFF
1,Offline,OFF
2,Offline,OFF
3,Offline,OFF
4,Offline,OFF


In [45]:
# Prefer vectorized ops (np.where) over apply/lambda when possible — much faster on large data
df_clean["high_value_student"] = np.where(df_clean["net_fee"] > 9000, "Yes", "No")
df_clean["high_value_student"].value_counts()

,count
high_value_student,
No,118
Yes,62


---
## ✅ Recap

| # | Topic | Key takeaway |
|---|---|---|
| 1 | Creating DataFrames | dict → DataFrame is the go-to |
| 2 | Reading CSV/Excel | `parse_dates`, `usecols`, `sheet_name` save cleanup work later |
| 3 | Selecting | `[]` for columns, `.loc`/`.iloc` for label/position rows |
| 4 | Filtering | boolean masks + `.isin()` + `.query()` |
| 5 | Sorting | `sort_values()`, multi-column with `ascending=[...]` |
| 6 | Missing Values | `isna()` to find, `fillna()`/`dropna()` to handle — think before filling |
| 7 | GroupBy | `agg()` with named outputs is the cleanest pattern |
| 8 | Merge & Join | `merge(..., how='left')` to enrich without losing rows |
| 9 | Pivot Tables | `pivot_table` + `margins=True` for quick cross-tab summaries |
| 10 | String Ops | `.str` accessor — always clean text data before analysis |
| 11 | DateTime | `.dt` accessor + `to_period()` for time-based grouping |
| 12 | Apply & Lambda | powerful but slower — vectorize (`np.where`) when you can |

**Dataset recap:** `course_enrollments.csv` (180 rows, messy on purpose) + `instructors.csv` (lookup table).

🎬 *Content idea: each section above is self-contained — one code cell ≈ one 30-60s reel/short. Screen-record the cell running + narrate the cheat-sheet table.*
